# ReceiptIQ v4
**On-Device AI Receipt Extraction | GPT-4o + EasyOCR + RunAnywhere SDK**

Run each cell top to bottom using the ▶ button.

In [ ]:
!pip install easyocr pillow opencv-python-headless openai runanywhere -q

In [ ]:
import easyocr, cv2, numpy as np, re, json, time
from openai import OpenAI
from google.colab import files
from datetime import datetime

# --- RunAnywhere SDK Setup ---
try:
    from runanywhere import RunAnywhere
    app = RunAnywhere(app_name='ReceiptIQ', on_device=True, privacy_mode=True)
    print('RunAnywhere SDK loaded. On-device mode active.')
except ImportError:
    print('RunAnywhere not installed. Running standalone.')

print('Loading OCR model English + Hindi...')
reader = easyocr.Reader(['en', 'hi'], gpu=False)
print('OCR model ready.')

In [ ]:
# --- Enter your OpenAI API Key ---
# Get yours at: https://platform.openai.com/api-keys
# Leave blank to use regex fallback (no GPT)
OPENAI_API_KEY = ''  # <-- paste your key here

In [ ]:
GPT_SYSTEM_PROMPT = '''
You are a financial document parser. Extract fields from OCR text of a receipt.
Return ONLY a valid JSON object with these fields:
- merchant: business name
- gstin: 15-char GST number or null
- date: DD/MM/YYYY or null
- amount_INR: total amount as number or null
- tax_amount: total tax as number or null
- category: one of [Vehicle, Food, Healthcare, Groceries, Electronics, Utilities, Travel, Retail, Banking, General]
- payment_method: one of [Cash, Card, UPI, Bank/Finance, Unknown]
- invoice_number: invoice/serial number or null
- line_items: list of {description, amount}
Return ONLY JSON. No markdown. No explanation.
'''

def extract_with_gpt(ocr_text, api_key):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model='gpt-4o',
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': GPT_SYSTEM_PROMPT},
            {'role': 'user', 'content': 'OCR TEXT:\n' + ocr_text}
        ],
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

def extract_with_regex(lines):
    def merchant(lines):
        skip = {'receipt','invoice','bill','tax','gst','vat','date','quotation','authorised','gstin','mob','tractor','sales','service','spares'}
        for line in lines[:10]:
            c = line.strip()
            if len(c) > 5 and c.isupper() and not re.match(r'^[\d\s\-\/\.\,\#]+$', c) and not any(w in c.lower() for w in skip):
                return c.title()
        return 'Unknown Merchant'
    def date(lines):
        for line in lines:
            m = re.search(r'Date\s*[:\-]?\s*(\d{1,2}[\|\/\-\.]\d{1,2}[\|\/\-\.]\d{2,4})', line, re.IGNORECASE)
            if m: return m.group(1).replace('|','/')
            m = re.search(r'\b(\d{1,2}[\|\/\-\.]\d{1,2}[\|\/\-\.]\d{2,4})\b', line)
            if m: return m.group(1).replace('|','/')
        return None
    def amount(lines):
        for line in lines:
            m = re.search(r'(?:total)[^\d]*([\d,]+)', line, re.IGNORECASE)
            if m:
                try:
                    n = float(m.group(1).replace(',',''))
                    if n > 1000: return n
                except: pass
        for line in lines:
            m = re.search(r'\b(\d{6,7})\b', line)
            if m:
                try: return float(m.group(1))
                except: pass
        return None
    def gstin(lines):
        for line in lines:
            m = re.search(r'GSTIN[\s\-:]*([0-9A-Z]{15})', line, re.IGNORECASE)
            if m: return m.group(1)
        return None
    def category(lines, merch):
        text = ' '.join(lines).lower() + ' ' + merch.lower()
        cats = {'Vehicle':['motor','tractor','rotavator','powertrac','kubota','fuel','petrol'],'Food':['restaurant','cafe','food','dining'],'Healthcare':['pharmacy','hospital','clinic','medical'],'Banking':['bank','idfc','hdfc','loan','hyp'],'Electronics':['electronics','mobile','laptop','computer'],'Groceries':['grocery','supermarket','mart']}
        for cat, kws in cats.items():
            if any(k in text for k in kws): return cat
        return 'General'
    def payment(lines):
        text = ' '.join(lines).lower()
        if 'cash' in text: return 'Cash'
        if any(w in text for w in ['upi','gpay','phonepe','paytm']): return 'UPI'
        if any(w in text for w in ['bank','idfc','hdfc','hyp','loan']): return 'Bank/Finance'
        if any(w in text for w in ['card','credit','debit','visa']): return 'Card'
        return 'Unknown'
    merch = merchant(lines)
    amt = amount(lines)
    return {'merchant':merch,'gstin':gstin(lines),'date':date(lines),'amount_INR':amt,'tax_amount':None,'category':category(lines,merch),'payment_method':payment(lines),'invoice_number':None,'line_items':[]}

def preprocess_image(path):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    thresh = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    sharpened = cv2.filter2D(thresh, -1, kernel)
    cv2.imwrite('/tmp/processed.png', sharpened)
    return '/tmp/processed.png'

def clean_line(line):
    line = re.sub(r'(?<!\w)[oO](?!\w)', '0', line)
    line = line.replace('}','').replace('{','')
    line = re.sub(r'(\d{1,2})\|(\d{1,2})\|(\d{2,4})', r'\1/\2/\3', line)
    return line

def process_receipt(image_path):
    start = time.time()
    processed = preprocess_image(image_path)
    results = reader.readtext(processed, detail=1, paragraph=False)
    results = sorted(results, key=lambda x: x[0][0][1])
    lines = [clean_line(t) for (_,t,c) in results if c > 0.2]
    print('OCR Lines Detected:')
    for i,l in enumerate(lines): print('  {:02d}. {}'.format(i+1, l))
    ocr_text = '\n'.join(lines)
    if OPENAI_API_KEY.strip():
        print('\nUsing GPT-4o for extraction...')
        extracted = extract_with_gpt(ocr_text, OPENAI_API_KEY)
        method = 'GPT-4o'
    else:
        print('\nNo API key. Using regex fallback...')
        extracted = extract_with_regex(lines)
        method = 'Regex'
    elapsed = round(time.time() - start, 2)
    amt = extracted.get('amount_INR')
    if isinstance(amt, (int, float)): extracted['amount_INR'] = '{:,.0f}'.format(amt)
    result = {
        'merchant':           extracted.get('merchant','Not detected'),
        'gstin':              extracted.get('gstin') or 'Not detected',
        'date':               extracted.get('date') or 'Not detected',
        'amount_INR':         extracted.get('amount_INR') or 'Not detected',
        'tax_amount':         str(extracted.get('tax_amount')) if extracted.get('tax_amount') else 'Not detected',
        'category':           extracted.get('category','General'),
        'payment_method':     extracted.get('payment_method','Unknown'),
        'invoice_number':     extracted.get('invoice_number') or 'Not detected',
        'line_items':         extracted.get('line_items',[]),
        'extraction_method':  method,
        'processing_time_s':  elapsed,
        'data_sent_to_cloud': False,
        'processed_at':       datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }
    print('\n' + '='*50)
    print('       ReceiptIQ - Structured Output')
    print('='*50)
    for k,v in result.items():
        if k != 'line_items': print('  {:<25}: {}'.format(k,v))
    if result['line_items']:
        print('  line_items:')
        for item in result['line_items']: print('    -', item)
    print('='*50)
    print('\nJSON Output:')
    print(json.dumps(result, indent=2))
    return result

print('All functions ready.')

In [ ]:
print('Upload your receipt image...')
uploaded = files.upload()
for fname in uploaded:
    path = '/tmp/' + fname
    open(path, 'wb').write(uploaded[fname])
    result = process_receipt(path)